# Run the site scraper on Google Colab

This notebook installs Scrapy + Playwright and runs one of the three spiders in this repo (`oecd`, `agendastad`, `elkeregiotelt`).

**How to use:** `Runtime` → `Run all`, then scroll to the bottom to download the results, **or** run the cells one by one and stop at the spider you care about.

**Keep this tab open** while a crawl runs — Colab disconnects idle sessions after ~90 min.

## 1. Install dependencies

In [ ]:
!pip install -q scrapy scrapy-playwright markdownify beautifulsoup4 lxml Pillow

In [ ]:
# Chromium + the Linux libs it needs. `install-deps` uses apt under the hood;
# Colab runs the kernel as root so no sudo is needed.
!playwright install chromium
!playwright install-deps chromium

## 2. Get the code

Clones the feature branch if it isn't there yet, otherwise pulls the latest.

In [ ]:
%%bash
set -euo pipefail
REPO_DIR=/content/google-scholar-scrapy-spider
BRANCH=claude/build-website-scraper-AOTSB
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --branch "$BRANCH" https://github.com/GdeJoode/google-scholar-scrapy-spider.git "$REPO_DIR"
else
  git -C "$REPO_DIR" fetch origin "$BRANCH"
  git -C "$REPO_DIR" checkout "$BRANCH"
  git -C "$REPO_DIR" pull --ff-only origin "$BRANCH"
fi

In [ ]:
%cd /content/google-scholar-scrapy-spider
%env SCRAPY_PROJECT=site

## 3. (Optional) Persist output to Google Drive

Colab's `/content/` storage is wiped when the session ends. Mount Drive and symlink `output/` into it to keep the crawl results. Skip this cell if you'd rather download a zip at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/scraper_output
!ln -sfn /content/drive/MyDrive/scraper_output output
!ls -la output

## 4. Test run (25 pages)

Always start with this — it confirms Playwright works and gives you a chance to inspect the output layout before committing to a full crawl.

In [ ]:
!scrapy crawl agendastad -s CLOSESPIDER_PAGECOUNT=25

In [ ]:
# Quick look at what was produced.
!ls -la output/agendastad/
!wc -l output/agendastad/pages.jsonl output/agendastad/publications.jsonl output/agendastad/downloads_manifest.jsonl 2>/dev/null
!head -n 1 output/agendastad/pages.jsonl | python -m json.tool | head -n 30

## 5. Full crawls

Uncomment / run only the spider(s) you want. These can take a while (potentially hours for `oecd`). Keep the tab open.

In [ ]:
# agendastad.nl (root site + one domain deep into each city-deal's own site)
!scrapy crawl agendastad

In [ ]:
# elkeregiotelt.nl (root site + one domain deep into each regiodeal / NPVR site)
!scrapy crawl elkeregiotelt

In [ ]:
# oecd.ai/en/ — excluding /en/data. This one is the largest; expect hours.
!scrapy crawl oecd

## 6. Download the output

If you did not mount Drive, run this to zip the output and grab it via the browser. Skip if you symlinked to Drive — the files are already in `MyDrive/scraper_output`.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/scraper_output', 'zip', root_dir='output')
print('Wrote', archive)
files.download(archive)